<p align="center">
    <span style="font-size:2.5em; font-weight:bold;">
        eFleetPlan - Optimal infrastructure and fleet operation of electric LCV
    </span>
</p>

<p align="center">
    <span style="font-size:1.5em; font-weight:bold;">
        Carolina Gil Ribeiro, Jagruti Thakur
    </span>
</p>

## 0. Importing dependencies

In [ ]:
import sys, os, time
from pathlib import Path
from glob import glob

# ── Project root detection ─────────────────────────────────────────────────
def _find_project_root(start: Path) -> Path:
    for parent in [start, *start.parents]:
        if (parent / "src").exists():           return parent
        if (parent / "pyproject.toml").exists(): return parent
        if (parent / "setup.py").exists():       return parent
    raise FileNotFoundError(
        f"Could not find project root from '{start}'. "
        "Expected 'src/', 'pyproject.toml', or 'setup.py' at the root."
    )

project_root = _find_project_root(Path(os.getcwd()))
sys.path.insert(0, str(project_root / "src"))
sys.path.insert(0, str(project_root))

# ── Imports ────────────────────────────────────────────────────────────────
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from config._0_supportfiles.config_loader_optimisation import (
    load_opt_config, RunOptConfig, InfrastructureConfig,
)
from src.efleetplan._2_co_optimisation.co_optimisation import optimisation, save_results
from src.efleetplan._2_co_optimisation.optimisation_graphs import (
    graph_vehicles, plot_summary_table, process_folder,
    graph_number_of_chargers_by_schedules, graph_chargingenergy, graph_energybytype,
)

print("All imports OK")


# 2. Charging Infrastructure Co-optimisation

## 2.1 Optimisation configuration

Edit `run_Optimisation_Config.yaml` and `infrastructure_configuration.yaml` to change parameters.

In [ ]:
# ── Load optimisation configuration ───────────────────────────────────────
# Edit run_Optimisation_Config.yaml and infrastructure_configuration.yaml to
# change parameters. Set infrastructure_configurations: custom in the run
# YAML to override infrastructure parameters inline.

config_dir = os.path.join(project_root, "config")

opt_config, cost_config, power_charge_config, run = load_opt_config(
    run_yaml   = os.path.join(config_dir, "run_Optimisation_Config.yaml"),
    env_yaml   = os.path.join(config_dir, "env.yaml"),
    infra_yaml = os.path.join(config_dir, "predefined", "infrastructure_configuration.yaml"),
)

schedule_name   = run.schedule_name
schedule_number = run.schedule_number

print("Configuration loaded successfully:")
print(f"  Schedule:   {schedule_name} (#{schedule_number})")
print(f"  Fleet size: {run.EVs} vehicles")
print(f"  MIP Gap:    {run.MIPGap}")


### 2.1.2 Cost and power parameters

In [ ]:
# ── Power and cost parameters ──────────────────────────────────────────────
print("--- Charger Power Parameters ---")
for k, v in power_charge_config.items():
    print(f"  {k}: {v}")

print("\n--- Infrastructure Cost Parameters ---")
for k, v in cost_config.items():
    print(f"  {k}: {v}")


### 2.1.3 Input data inspection

In [ ]:
# ── Inspect input data shape ───────────────────────────────────────────────
for key in ["En_consumption", "Ev_distance", "EV_availability",
            "Battery_Limitation", "PowerRate_Limitation"]:
    val = opt_config[key]
    print(f"{key}: shape={val.shape},  index dtype={val.index.dtype}")


## 2.2 Run optimisation

In [ ]:
# ── Run optimisation ───────────────────────────────────────────────────────
m, Price, EV_availability, Distance_km = optimisation(opt_config, cost_config, power_charge_config)


## 2.3 Save results

In [ ]:
# ── Save results ───────────────────────────────────────────────────────────
results_folder = os.path.join(project_root, "data", "Output", schedule_name, "Results")
os.makedirs(results_folder, exist_ok=True)

csv_file_pathA = os.path.join(results_folder, f"{schedule_number}_Main_variables_results.csv")
csv_file_pathB = os.path.join(results_folder, f"{schedule_number}_results_summary.csv")
csv_file_pathC = os.path.join(results_folder, f"{schedule_number}_results_per_EV.csv")

save_results(
    m, Price, EV_availability, Distance_km,
    csv_file_pathA, csv_file_pathB, csv_file_pathC,
    cost_config, power_charge_config,
)
print("Results saved to:", results_folder)


## 2.4 Post-processing and visualisation

In [ ]:
# ── Summary table ─────────────────────────────────────────────────────────
file_pattern = os.path.join(project_root, "data", "Output", schedule_name, "Results", "*_results_summary.csv")
matched = glob(file_pattern)
if not matched:
    raise FileNotFoundError(f"No file matches: {file_pattern}")
plot_summary_table(matched[0])


### Compute aggregates (max / average)

In [ ]:
# ── Compute max / average aggregates ──────────────────────────────────────
folder_path = os.path.join(project_root, "data", "Output", schedule_name, "Results")
filename_pattern = os.path.join(folder_path, "*_Main_variables_results.csv")
process_folder(folder_path, filename_pattern)


### Graph 1 — Number of chargers by schedule

In [ ]:
# ── Graph: number of chargers by schedule ─────────────────────────────────
folder_path     = os.path.join(project_root, "data", "Output", schedule_name, "Results")
files_pertime_max = glob(os.path.join(folder_path, "*_max_variable_per_step.csv"))
graph_number_of_chargers_by_schedules(folder_path, files_pertime_max)


### Graph 2 — Average charging power and electricity price

In [ ]:
# ── Graph: charging power and electricity price ────────────────────────────
folder_path = os.path.join(project_root, "data", "Output", schedule_name, "Results")
matched     = glob(os.path.join(folder_path, "*_avg_variable_per_step.csv"))
if not matched:
    raise FileNotFoundError("Average variable file not found — run process_folder first.")
graph_chargingenergy(matched[0], folder_path)
graph_energybytype(matched[0], folder_path)


### Graph 3 — Per-vehicle charging, discharging power and SOC

In [ ]:
# ── Graph: per-vehicle charging, discharging and SOC ──────────────────────
n_vehicles = 3   # number of vehicles to plot
n_days     = 4   # number of days to plot

folder_path = os.path.join(project_root, "data", "Output", schedule_name, "Results")
graph_vehicles(folder_path, csv_file_pathC, n_days, n_vehicles)
